In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

DATA_DIR = Path('../PS3/02_Datasets/SHM')
TEST_DIR = DATA_DIR / 'Test'
MODEL_PATH = Path('shm_final_model.joblib')
FEATURE_PATH = Path('shm_final_features.csv')
TEST_FILES = sorted(TEST_DIR.glob('test*.csv'))
print('Test files:', len(TEST_FILES))
print('Model exists:', MODEL_PATH.exists())
print('Feature list exists:', FEATURE_PATH.exists())
if len(TEST_FILES) != 16: print('Warning: expected 16 test files.')

Test files: 16
Model exists: True
Feature list exists: True


In [2]:
def load_signal(path):
    return pd.read_csv(path, usecols=[0]).iloc[:, 0].to_numpy(dtype=np.float64)

def safe_quantile(x, q): return float(np.quantile(x, q))

def zero_crossing_rate(x):
    centred = x - np.mean(x)
    return float(np.mean(centred[:-1] * centred[1:] < 0))

def spectral_features(x):
    max_points = 200000
    if len(x) > max_points:
        x = x[np.linspace(0, len(x)-1, max_points).astype(int)]
    x = x - np.mean(x)
    power = np.abs(np.fft.rfft(x)) ** 2
    if len(power) <= 1 or power.sum() == 0:
        return {'fft_dominant_bin':0.0,'fft_spectral_centroid':0.0,'fft_low_power_ratio':0.0,'fft_high_power_ratio':0.0}
    freq = np.fft.rfftfreq(len(x), d=1.0); power[0] = 0.0; total = power.sum(); split = max(1, len(power)//4)
    return {'fft_dominant_bin':float(np.argmax(power)), 'fft_spectral_centroid':float((freq*power).sum()/total), 'fft_low_power_ratio':float(power[:split].sum()/total), 'fft_high_power_ratio':float(power[-split:].sum()/total)}

def extract_features(x, filename):
    x = np.asarray(x, dtype=np.float64); x = x[np.isfinite(x)]
    if len(x) == 0: raise ValueError(f'No valid samples in {filename}')
    mean = np.mean(x); std = np.std(x); abs_x = np.abs(x); rms = np.sqrt(np.mean(x*x)); diff = np.diff(x)
    out = {'filename':filename,'n_samples':len(x),'mean':mean,'std':std,'min':np.min(x),'max':np.max(x),'range':np.ptp(x),'median':np.median(x),'q01':safe_quantile(x,.01),'q05':safe_quantile(x,.05),'q25':safe_quantile(x,.25),'q75':safe_quantile(x,.75),'q95':safe_quantile(x,.95),'q99':safe_quantile(x,.99),'iqr':safe_quantile(x,.75)-safe_quantile(x,.25),'abs_mean':np.mean(abs_x),'rms':rms,'energy_per_sample':np.mean(x*x),'crest_factor':np.max(abs_x)/(rms+1e-12),'zero_crossing_rate':zero_crossing_rate(x),'diff_mean':np.mean(diff),'diff_std':np.std(diff),'diff_abs_mean':np.mean(np.abs(diff)),'diff_max_abs':np.max(np.abs(diff))}
    out.update(spectral_features(x)); return out

In [3]:
model = joblib.load(MODEL_PATH)
feature_columns = pd.read_csv(FEATURE_PATH)['feature'].tolist()
rows = []
for i, path in enumerate(TEST_FILES, 1):
    rows.append(extract_features(load_signal(path), path.name))
    print(f'Processed {i}/{len(TEST_FILES)}: {path.name}')
test_features = pd.DataFrame(rows)
missing = [c for c in feature_columns if c not in test_features.columns]
extra = [c for c in test_features.columns if c not in feature_columns + ['filename']]
print('Missing features:', missing)
print('Extra features:', extra)
if missing: raise ValueError('Test feature schema does not match the final model.')
X_test = test_features[feature_columns].replace([np.inf,-np.inf], np.nan)
if X_test.isna().any().any(): raise ValueError('Test features contain missing or invalid values.')
display(X_test.head())

Processed 1/16: test01.csv
Processed 2/16: test02.csv
Processed 3/16: test03.csv
Processed 4/16: test04.csv
Processed 5/16: test05.csv
Processed 6/16: test06.csv
Processed 7/16: test07.csv
Processed 8/16: test08.csv
Processed 9/16: test09.csv
Processed 10/16: test10.csv
Processed 11/16: test11.csv
Processed 12/16: test12.csv
Processed 13/16: test13.csv
Processed 14/16: test14.csv
Processed 15/16: test15.csv
Processed 16/16: test16.csv
Missing features: []
Extra features: []


,n_samples,mean,std,min,max,range,median,q01,q05,q25,...,crest_factor,zero_crossing_rate,diff_mean,diff_std,diff_abs_mean,diff_max_abs,fft_dominant_bin,fft_spectral_centroid,fft_low_power_ratio,fft_high_power_ratio
0,581119,7.117228,5.007661,-19.305857,29.151978,48.457835,7.352967,-5.334422,-1.524804,3.475320,...,3.349881,0.031291,0.000004,0.731269,0.580507,7.164293,3.0,0.005284,0.986992,0.003174
1,581119,-14.257164,15.367175,-61.985485,31.014269,92.999754,-15.075310,-47.855969,-42.150810,-21.265938,...,2.957002,0.019629,-0.000034,0.729099,0.567007,8.618410,6.0,0.000920,0.997776,0.000254
2,581119,-4.617882,10.748640,-55.626373,40.574986,96.201359,-5.110671,-32.135147,-25.408792,-9.683488,...,4.754945,0.045850,-0.000003,0.611424,0.453537,12.053869,5.0,0.001445,0.996508,0.000552
3,581119,3.184893,4.792224,-16.517735,27.199333,43.717068,2.351780,-6.419695,-3.915382,1.161274,...,4.726999,0.047452,0.000010,0.631722,0.496107,5.421052,5.0,0.004007,0.990657,0.002294
4,581119,-0.630171,3.698369,-21.497131,20.242847,41.739978,-0.467699,-8.703446,-6.751868,-3.065552,...,5.730013,0.066582,0.000001,0.740475,0.586436,8.673684,6.0,0.009514,0.975392,0.004027


In [4]:
predictions = model.predict(X_test)
# Damage is non-negative by definition; clip only numerical underflow below zero.
predictions = np.maximum(predictions, 0.0)
submission = pd.DataFrame({'file_id':test_features['filename'], 'prediction':predictions})
display(submission)
display(submission['prediction'].describe())

,file_id,prediction
0,test01.csv,0.062485
1,test02.csv,0.813850
2,test03.csv,0.775235
3,test04.csv,0.037684
4,test05.csv,0.045738
5,test06.csv,0.441940
6,test07.csv,0.226028
7,test08.csv,0.072227
8,test09.csv,0.085802
9,test10.csv,0.067339


count    16.000000
mean      0.243838
std       0.261474
min       0.037684
25%       0.060197
50%       0.079015
75%       0.425458
max       0.813850
Name: prediction, dtype: float64

In [5]:
if submission['file_id'].duplicated().any(): raise ValueError('Duplicate test file IDs found.')
if submission['prediction'].isna().any(): raise ValueError('Missing predictions found.')
if len(submission) != len(TEST_FILES): raise ValueError('Prediction count does not match test file count.')
assert submission.columns.tolist() == ['file_id','prediction']
submission.to_csv('shm_predictions.csv', index=False)
print('Saved:', Path('shm_predictions.csv').resolve())
print('Rows:', len(submission))

Saved: C:\Users\limju\Documents\AY2627\Semester 1\Hackathon\shm_predictions.csv
Rows: 16
